In [ ]:
#  The MIT License (MIT)
#
#  Copyright (c) 2015-2026 Advanced Micro Devices, Inc. All rights reserved.
#
#  Permission is hereby granted, free of charge, to any person obtaining a copy
#  of this software and associated documentation files (the 'Software'), to deal
#  in the Software without restriction, including without limitation the rights
#  to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
#  copies of the Software, and to permit persons to whom the Software is
#  furnished to do so, subject to the following conditions:
#
#  The above copyright notice and this permission notice shall be included in
#  all copies or substantial portions of the Software.
#
#  THE SOFTWARE IS PROVIDED 'AS IS', WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
#  IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
#  FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.  IN NO EVENT SHALL THE
#  AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
#  LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
#  OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
#  THE SOFTWARE.

# Object Detection with Yolo26

This notebook demonstrates how to export a YOLO26 model from Ultralytics to ONNX, compile it with AMD MIGraphX (applying FP16 quantization), and run inference on a sample image.

## Prerequisites
- AMD GPU with ROCm installed (https://rocm.docs.amd.com/en/docs-7.2.0/)

In [ ]:
%pip install -q numpy opencv-python Pillow
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/rocm7.2
%pip install -q ultralytics

- To use MIGraphX's Python module, you have to add "/opt/rocm/lib/" to the search module path.

In [ ]:
import sys
sys.path.append('/opt/rocm/lib/')

- Check GPU device

In [ ]:
import cv2
import numpy as np
import torch
import migraphx
import requests
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Configuration
Set the model parameters. We use `yolo26x` by default

In [ ]:
MODEL_NAME = "yolo26x"  # Can be yolo26n, s, m, l, x
BATCH_SIZE = 1
IMGSZ = [640, 640]  # Height, Width
CONF_THRES = 0.25

## 1. Export Model to ONNX
We load the YOLO model using Ultralytics and export it to ONNX format.

In [ ]:
# Load the YOLO model (will download if not found)
model = YOLO(f"{MODEL_NAME}.pt")

# Export to ONNX
onnx_model_path = model.export(format="onnx", dynamic=False, imgsz=IMGSZ, batch=BATCH_SIZE)
print(f"Model exported to: {onnx_model_path}")

## 2. Compile with MIGraphX
Now we compile the ONNX model using MIGraphX, applying FP16 quantization for performance.

In [ ]:
print("MIGraphX: Loading ONNX model...")
mxr_model = migraphx.parse_onnx(onnx_model_path)

print("MIGraphX: Applying FP16 quantization...")
migraphx.quantize_fp16(mxr_model)

print("MIGraphX: Compilation...")
mxr_model.compile(t=migraphx.get_target("gpu"))

print("MIGraphX: Saving compiled model...")
mxr_model_path = onnx_model_path.replace(".onnx", ".mxr")
migraphx.save(mxr_model, mxr_model_path)

## 3. Preprocessing Helper
We need to preprocess the input image to match the model's expected input format (Resize, Pad, Normalize, HWC->CHW).

In [ ]:
def preprocess_image(image_path, input_size):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    # Letterbox resize
    h, w = img.shape[:2]
    target_h, target_w = input_size
    scale = min(target_h / h, target_w / w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))

    img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    pad_x, pad_y = (target_w - new_w) // 2, (target_h - new_h) // 2
    left, right, top, bottom = (pad_x, target_w - new_w - pad_x, 
                                pad_y, target_h - new_h - pad_y)
    img_padded = cv2.copyMakeBorder(img_resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))

    # Convert
    img_in = img_padded.transpose((2, 0, 1))[::-1]  # HWC to CHW, BGR to RGB
    img_in = np.ascontiguousarray(img_in)
    
    # Normalize
    img_in = img_in.astype(np.float32) / 255.0
    img_in = img_in[None]  # Add batch dimension

    return img_in, img, (h, w), (left, right, top, bottom), scale


## 4. Run Inference
Download a sample image and run inference using the compiled MIGraphX model.

In [ ]:
# Download sample image (bus.jpg) if it doesn't exist
image_url = "https://ultralytics.com/images/bus.jpg"
image_path = "bus.jpg"
if not Path(image_path).exists():
    print(f"Downloading {image_path}...")
    with open(image_path, "wb") as f:
        f.write(requests.get(image_url).content)

# Preprocess
input_tensor, original_image, original_shape, padding, scale = preprocess_image(image_path, IMGSZ)

# Get input and output names and convert input tensor to an MIGraphX argument
input_name = next(iter(mxr_model.get_parameter_shapes()))
input_argument = migraphx.argument(input_tensor)

# Run inference
outputs = mxr_model.run({input_name: input_argument})
print("Inference complete.")

## 6. Postprocessing and Visualization
Filter detections with low confidence, becasue yolo26 is nms free model, and draw bounding boxes on the original image.

In [ ]:
# MIGraphX run() returns a list of outputs; we take the first one.
# YOLO26 end-to-end head returns shape [1, 300, 6] = [batch, num_boxes, (x1, y1, x2, y2, conf, cls)]
predictions = np.array(outputs[0])

# Get the first batch, shape becomes [300, 6]
predictions = predictions[0]

# Filter detections by confidence threshold
detections = predictions[predictions[:, 4] > CONF_THRES]

# Copy original image for drawing
img_result = original_image.copy()

if len(detections) > 0:
    for det in detections:
        x1, y1, x2, y2, conf, cls_id = det
        cls_id = int(cls_id)
        
        # 1. Reverse the padding applied during preprocessing
        x1 -= padding[0]
        y1 -= padding[2]
        x2 -= padding[1]
        y2 -= padding[3]
        
        # 2. Reverse the scaling applied during preprocessing
        x1 /= scale
        y1 /= scale
        x2 /= scale
        y2 /= scale
        
        # 3. Clip bounding boxes to the original image dimensions
        h, w = original_shape
        x1 = int(np.clip(x1, 0, w))
        y1 = int(np.clip(y1, 0, h))
        x2 = int(np.clip(x2, 0, w))
        y2 = int(np.clip(y2, 0, h))
        
        # 4. Draw the bounding box
        cv2.rectangle(img_result, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # 5. Draw the label
        label = f"{model.names[cls_id]} {conf:.2f}"
        (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img_result, (x1, y1 - label_h - 5), (x1 + label_w, y1), (0, 255, 0), -1)
        cv2.putText(img_result, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
else:
    print("No detections found.")

Image.fromarray(img_result)